# Airbnb Paris – Exp 2: Enhanced Baseline-Modelle
- Gleiche Baselines auf verschiedenen Repräsentationen: cleaned vs. semantisch (PCA/no PCA) vs. enhanced (PCA/no PCA) vs. enhanced+semantisch
- Alignment über `row_id`, gemeinsamer 70/30-Split; feste Detektor-Params (fairer Vergleich)

In [7]:
import time
import numpy as np
import pandas as pd
import mlflow
from sklearn.model_selection import train_test_split
from sklearn.metrics import average_precision_score, roc_auc_score
from pyod.models.iforest import IForest
from pyod.models.loda import LODA
from pyod.models.ecod import ECOD
from pyod.models.auto_encoder import AutoEncoder

## Repräsentationen laden (indexiert über row_id)
- Outlier = `is_top_rating == 0`; enhanced+semantisch = PCA30-Konkatenation

In [8]:
LABEL = "is_top_rating"
ds = "airbnb_paris"

def_load = lambda name: pd.read_csv(f"../../data/preprocessed/{name}_{ds}.csv").set_index("row_id")
cleaned = def_load("cleaned")
semantic_pca100 = def_load("semantic_pca100")
semantic_pca = def_load("semantic_pca30")
enhanced = def_load("enhanced")
enhanced_pca = def_load("enhanced_pca30")

reps = {
    "cleaned": cleaned.drop(columns=[LABEL]),
    "semantic_pca100": semantic_pca100.drop(columns=[LABEL]),
    "semantic_pca30": semantic_pca.drop(columns=[LABEL]),
    "enhanced": enhanced.drop(columns=[LABEL]),
    "enhanced_pca30": enhanced_pca.drop(columns=[LABEL]),
    "enhanced_semantic_pca30": enhanced_pca.drop(columns=[LABEL]).join(
        semantic_pca.drop(columns=[LABEL]), how="inner", lsuffix="_enh", rsuffix="_sem"),
}

## Gemeinsamer Index & Split
- Schnittmenge aller Repräsentationen (robust gegen unvollständige semantic-CSVs)

In [9]:
common = cleaned.index
for r in reps.values():
    common = common.intersection(r.index)
common = common.sort_values()
y = (1 - cleaned.loc[common, LABEL]).values
print("common rows:", len(common), "outlier rate", round(y.mean(), 4))

tr_id, te_id = train_test_split(common, test_size=0.3, stratify=y, random_state=42)
y_train = (1 - cleaned.loc[tr_id, LABEL]).values
y_test = (1 - cleaned.loc[te_id, LABEL]).values

mlflow.set_tracking_uri("file:../../mlruns")
mlflow.set_experiment("airbnb_paris_experiment_2")

common rows: 18350 outlier rate 0.0393


<Experiment: artifact_location='file:///home/debian/TFM_master_thesis/airbnb_notebooks/exp2/../../mlruns/471299177397327356', creation_time=1780334063406, experiment_id='471299177397327356', last_update_time=1780334063406, lifecycle_stage='active', name='airbnb_paris_experiment_2', tags={}, trace_location=None, workspace='default'>

## Detektoren x Repräsentationen
- Beste Hyperparameter aus Exp 1 (README), **kein GridSearch**; AutoEncoder One-Class (nur Inlier, GPU)

In [10]:
# beste Hyperparameter aus Experiment 1 (README) — kein GridSearch in Exp 2
detectors = {
    "iforest": (IForest, {"n_estimators": 100, "max_features": 1.0, "random_state": 42}, False),
    "loda": (LODA, {"n_bins": 10, "n_random_cuts": 100}, False),
    "ecod": (ECOD, {}, False),
    "autoencoder": (AutoEncoder, {"hidden_neuron_list": [64, 32], "epoch_num": 50, "random_state": 42, "device": "cuda"}, True),
}

for rep_name, rep in reps.items():
    Xtr = rep.loc[tr_id].values
    Xte = rep.loc[te_id].values
    for det_name, (Model, params, inlier_only) in detectors.items():
        t0 = time.perf_counter()
        Xfit = Xtr[y_train == 0] if inlier_only else Xtr
        model = Model(**params)
        model.fit(Xfit)
        scores = model.decision_function(Xte)
        runtime = time.perf_counter() - t0
        ap = average_precision_score(y_test, scores)
        auc = roc_auc_score(y_test, scores)
        with mlflow.start_run(run_name=f"{rep_name}__{det_name}"):
            mlflow.log_param("representation", rep_name)
            mlflow.log_param("detector", det_name)
            mlflow.log_param("n_features", rep.shape[1])
            mlflow.log_metric("average_precision", ap)
            mlflow.log_metric("auc_roc", auc)
            mlflow.log_metric("runtime_s", runtime)
        print(f"{rep_name:24s} {det_name:12s} AP={ap:.4f} AUC={auc:.4f} feat={rep.shape[1]} t={runtime:.1f}s")

cleaned                  iforest      AP=0.0886 AUC=0.6962 feat=40 t=0.3s
cleaned                  loda         AP=0.0951 AUC=0.6825 feat=40 t=0.1s
cleaned                  ecod         AP=0.1085 AUC=0.7240 feat=40 t=0.1s


Training: 100%|██████████| 50/50 [00:48<00:00,  1.03it/s]


cleaned                  autoencoder  AP=0.0714 AUC=0.6263 feat=40 t=49.1s
semantic_pca100          iforest      AP=0.0698 AUC=0.6116 feat=154 t=0.3s
semantic_pca100          loda         AP=0.0514 AUC=0.5150 feat=154 t=0.2s
semantic_pca100          ecod         AP=0.0677 AUC=0.6141 feat=154 t=1.2s


Training: 100%|██████████| 50/50 [00:53<00:00,  1.07s/it]


semantic_pca100          autoencoder  AP=0.0512 AUC=0.5117 feat=154 t=54.3s
semantic_pca30           iforest      AP=0.0808 AUC=0.6613 feat=84 t=0.3s
semantic_pca30           loda         AP=0.0684 AUC=0.5937 feat=84 t=0.2s
semantic_pca30           ecod         AP=0.0846 AUC=0.6733 feat=84 t=0.3s


...

enhanced                 autoencoder  AP=0.3190 AUC=0.7830 feat=512 t=49.3s
enhanced_pca30           iforest      AP=0.2305 AUC=0.7569 feat=30 t=0.3s
enhanced_pca30           loda         AP=0.1937 AUC=0.7299 feat=30 t=0.1s
enhanced_pca30           ecod         AP=0.2705 AUC=0.7455 feat=30 t=0.1s


Training: 100%|██████████| 50/50 [00:47<00:00,  1.05it/s]


enhanced_pca30           autoencoder  AP=0.4692 AUC=0.8206 feat=30 t=48.3s
enhanced_semantic_pca30  iforest      AP=0.1029 AUC=0.7306 feat=114 t=0.3s
enhanced_semantic_pca30  loda         AP=0.0653 AUC=0.5384 feat=114 t=0.1s
enhanced_semantic_pca30  ecod         AP=0.1462 AUC=0.7620 feat=114 t=0.6s


Training: 100%|██████████| 50/50 [00:47<00:00,  1.04it/s]


enhanced_semantic_pca30  autoencoder  AP=0.1850 AUC=0.6931 feat=114 t=48.4s
